✅ Data profiling complete

📝 Final Silver rules (from profiling):

#	Dataset	Rule
1	eCommerce	Validate event_type ∈ {view, cart, purchase}
2	eCommerce	Null brand (13.94%) → "unknown"
3	eCommerce	Null category_code (32.21%) → "unknown"; split into category levels
4	eCommerce	Flag price = 0 (0.23%, no purchases) with is_zero_price
5	eCommerce	Convert event_time text → timestamp; add event_date and event_month
6	A/B	Exclude group/page mismatches (3,893 rows)
7	A/B	Deduplicate user 773192, keeping the first exposure → 290,584 rows
8	Instacart	Keep null days_since_prior_order; add is_first_order
9	Instacart	Exclude the 75,000 test orders
10	Instacart	Stack prior + train (composite key confirmed unique)
11	Instacart	Flatten aisles and departments into products

In [0]:
USE CATALOG dbw_kroger_insights;

In [0]:
SELECT
    event_type,
    COUNT(*) AS row_count,
    ???  AS pct_of_total
FROM bronze.ecommerce_events
GROUP BY event_type
ORDER BY row_count DESC;

In [0]:
SELECT
    round(SUM(CASE WHEN brand IS NULL THEN 1 ELSE 0 END)) AS null_brand_count,
    round(SUM(CASE WHEN brand IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) AS null_brand_pct,
    round(SUM(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END)) AS null_category_count,
    round(SUM(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) AS null_category_pct
FROM bronze.ecommerce_events;

In [0]:
SELECT
    MIN(price) AS min_price,
    MAX(price) AS max_price,
    SUM(CASE WHEN price <= 0 THEN 1 ELSE 0 END) AS zero_or_negative_count
FROM bronze.ecommerce_events;

In [0]:
SELECT
    event_type,
    COUNT(*) AS zero_price_count
FROM bronze.ecommerce_events
WHERE price = 0
GROUP BY event_type
ORDER BY zero_price_count DESC;

In [0]:
SELECT
    `group`,
    landing_page,
    COUNT(*) AS row_count
FROM bronze.ab_test
GROUP BY `group`, landing_page
ORDER BY row_count DESC;

In [0]:
WITH duplicate_users AS (
    SELECT user_id
    FROM bronze.ab_test
    GROUP BY user_id
    HAVING COUNT(*) > 1
)
SELECT COUNT(*) AS duplicate_user_count
FROM duplicate_users;

In [0]:
SELECT user_id
FROM bronze.ab_test
WHERE (`group` = 'treatment' AND landing_page = 'new_page')
   OR (`group` = 'control' AND landing_page = 'old_page')
GROUP BY user_id
HAVING COUNT(*) > 1;

In [0]:
SELECT *
FROM bronze.ab_test
WHERE user_id = 773192;

In [0]:
SELECT
    COUNT(*) AS null_days_since_prior_order,
    COUNT(DISTINCT user_id) AS distinct_users
FROM bronze.ic_orders
WHERE days_since_prior_order IS NULL;

In [0]:
SELECT
    eval_set,
    COUNT(*) AS order_count
FROM bronze.ic_orders
GROUP BY eval_set
ORDER BY eval_set;

In [0]:
SELECT
    order_id,
    product_id,
    COUNT(*) AS occurrence_count
FROM bronze.ic_order_products_prior
GROUP BY order_id, product_id
HAVING COUNT(*) > 1;

In [0]:
SELECT
    SUM(CASE WHEN days_since_prior_order IS NULL THEN 1 ELSE 0 END) AS null_rows,
    COUNT(DISTINCT CASE WHEN days_since_prior_order IS NULL THEN user_id END) AS users_with_null,
    COUNT(DISTINCT user_id) AS total_users
FROM bronze.ic_orders;